In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

MODELS = [
    'google_gemma-3-1b-pt',
    'google_gemma-3-270m',
    'meta-llama_Llama-3.2-1B',
    'EleutherAI_pythia-160m',
    'EleutherAI_pythia-410m',
    'EleutherAI_pythia-1b',
    'EleutherAI_pythia-1.4b',
    'EleutherAI_pythia-2.8b',
    'Qwen_Qwen2.5-0.5B',
    'Qwen_Qwen2.5-1.5B'
]   

def sigmoid(x, L, x0, k):
    return 1 / (1 + np.exp(-k * (x - x0)))

import os 

def _r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return np.nan if ss_tot == 0 else 1.0 - ss_res / ss_tot

def crossing_nearest(x, y, threshold=0.5, max_tol=0.1):
    if len(x) == 0:
        return np.nan
    idx = np.argmin(np.abs(y - threshold))
    if np.abs(y - threshold)[idx] > max_tol:
        return np.nan
    return float(x[idx])


def plot_accessibility_curves(model, fit_sigmoid=True, save=False):
    print(model)
    file_path_rand = f'runs/{model}/data_rnd_vocab_100k.csv'
    if not os.path.exists(file_path_rand):
        return None
    data_rand = pd.read_csv(file_path_rand)
    data = pd.read_csv(f'runs/{model}/data_dataset.csv')
    agg_dict = {
        'repeatability': 'mean',
        'curr_avg_repeatability': 'mean',
        'best_acc': 'mean',
        'num_steps': 'mean'
    }
    grouped_rand = data_rand.groupby(['n_mem', 'length']).agg(agg_dict).reset_index()
    grouped = data.groupby(['n_mem', 'length']).agg(agg_dict).reset_index()

    n_mems = sorted(grouped['n_mem'].unique())
    n_mems = [x for x in n_mems if x <=5]
    cmap = plt.get_cmap('Blues')
    cmap_rand = plt.get_cmap('Reds')
    colors = cmap(np.linspace(0.4, 0.9, max(1, len(n_mems))))
    colors_rand = cmap_rand(np.linspace(0.4, 0.9, max(1, len(n_mems))))

    fig = plt.figure(figsize=(12, 4))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.8, 1], height_ratios=[1, 1], wspace=0.2, hspace=0.3)
    ax_dataset = fig.add_subplot(gs[0, 0])
    ax_rand = fig.add_subplot(gs[1, 0], sharex=ax_dataset)
    ax_lin = fig.add_subplot(gs[:, 1])

    crossings = []
    crossings_rand = []

    # Dataset subplot (top-left)
    for i, n_mem in enumerate(n_mems):
        subset = grouped[grouped['n_mem'] == n_mem]
        if subset.empty:
            crossings.append((int(n_mem), np.nan))
            continue
        x_data = subset['length'].values
        y_data = subset['repeatability'].values

        x_cross = np.nan
        if fit_sigmoid and len(x_data) >= 3:
            p0 = [1.0, float(np.median(x_data)), 0.1]
            popt, _ = curve_fit(sigmoid, x_data, y_data, p0=p0, maxfev=10000)
            x_smooth = np.linspace(x_data.min(), x_data.max(), 200)
            y_smooth = sigmoid(x_smooth, *popt)
            y_fit = sigmoid(x_data, *popt)
            r2_fit = _r2(y_data, y_fit)
            print(f"{model} - Dataset n_mem={n_mem}: R²={r2_fit:.3f}")

            ax_dataset.plot(x_smooth, y_smooth, color=colors[i], linewidth=1, linestyle='-', alpha=1, label=n_mem)
            ax_dataset.plot(x_data, y_data, color=colors[i], linestyle=':', alpha=0.3)
            ax_dataset.scatter(x_data, y_data, color=colors[i], s=8, alpha=0.35)

            if popt[1] > 1000 or popt[1] < 0:
                x_cross = np.nan
            else:
                x_cross = float(popt[1])
                ax_dataset.axvline(x=x_cross, ymax=0.5, color=colors[i], linestyle='--', alpha=1, linewidth=1)
        else:
            ax_dataset.plot(x_data, y_data, label=str(n_mem), color=colors[i], markersize=4)
            x_cross = crossing_nearest(x_data, y_data, 0.5, max_tol=0.1)
            if not np.isnan(x_cross):
                ax_dataset.axvline(x=x_cross, ymax=0.5, color=colors[i], linestyle='--', alpha=1, linewidth=1)

        crossings.append((int(n_mem), float(x_cross) if not np.isnan(x_cross) else np.nan))

    # Random subplot (bottom-left)
    for i, n_mem in enumerate(n_mems):
        subset_r = grouped_rand[grouped_rand['n_mem'] == n_mem]
        if subset_r.empty:
            crossings_rand.append((int(n_mem), np.nan))
            continue
        x_data = subset_r['length'].values
        y_data = subset_r['repeatability'].values

        x_cross = np.nan
        if fit_sigmoid and len(x_data) >= 3:
            p0 = [1.0, float(np.median(x_data)), 0.1]
            popt, _ = curve_fit(sigmoid, x_data, y_data, p0=p0, maxfev=10000)
            x_smooth = np.linspace(x_data.min(), x_data.max(), 200)
            y_smooth = sigmoid(x_smooth, *popt)
            y_fit = sigmoid(x_data, *popt)

            r2_fit = _r2(y_data, y_fit)
            print(f"{model} - Random n_mem={n_mem}: R²={r2_fit:.3f}")

            ax_rand.plot(x_smooth, y_smooth, color=colors_rand[i], linewidth=1, linestyle='-', alpha=0.9, label=n_mem)
            ax_rand.plot(x_data, y_data, color=colors_rand[i], linestyle=':', alpha=0.3)
            ax_rand.scatter(x_data, y_data, color=colors_rand[i], s=8, alpha=0.35)

            if not (popt[1] > 1000 or popt[1] < 0):
                x_cross = float(popt[1])
                ax_rand.axvline(x=x_cross, ymax=0.5, color=colors_rand[i], linestyle='--', alpha=0.9, linewidth=1)
        else:
            ax_rand.plot(x_data, y_data, color=colors_rand[i], markersize=4)
            x_cross = crossing_nearest(x_data, y_data, 0.5, max_tol=0.1)
            if not np.isnan(x_cross):
                ax_rand.axvline(x=x_cross, ymax=0.5, color=colors_rand[i], linestyle='--', alpha=0.9, linewidth=1)

        crossings_rand.append((int(n_mem), float(x_cross) if not np.isnan(x_cross) else np.nan))

    # Styling for dataset subplot
    ax_dataset.set_ylabel('Acc. (PG19)', fontsize=10)
    ax_dataset.set_yticks([0.0, 0.5, 1.0])
    ax_dataset.set_title(f'{model.split("_")[-1].capitalize()}', fontsize=10)
    ax_dataset.set_ylim(-0.1, 1.1)
    ax_dataset.spines['top'].set_visible(False)
    ax_dataset.spines['right'].set_visible(False)
    ax_dataset.xaxis.set_ticks_position('bottom')
    ax_dataset.yaxis.set_ticks_position('left')
    ax_dataset.grid(False)
    legend_ds = ax_dataset.legend(fontsize=9, title='$m$ (tokens)', title_fontsize=9, loc='upper right')
    legend_ds.get_frame().set_linewidth(0.)
    ax_dataset.tick_params(labelbottom=False)  # share x-axis; hide top labels

    # Styling for random subplot
    ax_rand.set_xlabel('$n$ (tokens)', fontsize=10)
    ax_rand.set_ylabel('Acc. (Random)', fontsize=10)
    ax_rand.set_yticks([0.0, 0.5, 1.0])
    # ax_rand.set_title('Random', fontsize=10)
    ax_rand.set_ylim(-0.1, 1.1)
    ax_rand.spines['top'].set_visible(False)
    ax_rand.spines['right'].set_visible(False)
    ax_rand.xaxis.set_ticks_position('bottom')
    ax_rand.yaxis.set_ticks_position('left')
    ax_rand.grid(False)
    legend_rand = ax_rand.legend(fontsize=9, title='$m$ (tokens)', title_fontsize=9, loc='upper right')
    legend_rand.get_frame().set_linewidth(0.)


    # Right axis unchanged
    mems = np.array([m for m, cx in crossings])
    cxs = np.array([cx for m, cx in crossings], dtype=float)
    mask = ~np.isnan(cxs)
    mem_arr = mems[mask].astype(float)
    cross_arr = cxs[mask].astype(float)

    mems_r = np.array([m for m, cx in crossings_rand])
    cxs_r = np.array([cx for m, cx in crossings_rand], dtype=float)
    mask_r = ~np.isnan(cxs_r)
    mem_arr_r = mems_r[mask_r].astype(float)
    cross_arr_r = cxs_r[mask_r].astype(float)

    ax_lin.set_title('', fontsize=10)
    ax_lin.set_xlabel('$m$ (tokens)', fontsize=10)
    ax_lin.set_ylabel('$n$ at Accessibility=50%', fontsize=10)
    ax_lin.spines['top'].set_visible(False)
    ax_lin.spines['right'].set_visible(False)
    ax_lin.xaxis.set_ticks_position('bottom')
    ax_lin.yaxis.set_ticks_position('left')
    ax_lin.grid(False)

    if mem_arr.size > 0:
        cmap2 = plt.get_cmap('Blues')
        colors2 = cmap2(np.linspace(0.4, 0.9, len(mem_arr)))
        for i, (m, cx) in enumerate(zip(mem_arr, cross_arr)):
            ax_lin.scatter(m, cx, color=colors2[i], s=40, zorder=5, label=None)

    if mem_arr_r.size > 0:
        cmap3 = plt.get_cmap('Reds')
        colors3 = cmap3(np.linspace(0.4, 0.9, len(mem_arr_r)))
        for i, (m, cx) in enumerate(zip(mem_arr_r, cross_arr_r)):
            ax_lin.scatter(m, cx, color=colors3[i], s=40, marker='x', zorder=5, label=None)

    legends = []
    if mem_arr.size >= 2:
        slope, intercept = np.polyfit(mem_arr, cross_arr, 1)
        y_pred = slope * mem_arr + intercept
        ss_res = np.sum((cross_arr - y_pred) ** 2)
        ss_tot = np.sum((cross_arr - np.mean(cross_arr)) ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else np.nan

        xs = np.linspace(mem_arr.min() - 0.2, mem_arr.max() + 0.2, 200)
        ys = slope * xs + intercept
        l1, = ax_lin.plot(xs, ys, color='black', linewidth=1, linestyle='--', zorder=2)
        legends.append((l1, f'Dataset fit: $R^2$={r2:.3f}, a={slope:.2f}'))

    if mem_arr_r.size >= 2:
        slope_r, intercept_r = np.polyfit(mem_arr_r, cross_arr_r, 1)
        y_pred_r = slope_r * mem_arr_r + intercept_r
        ss_res_r = np.sum((cross_arr_r - y_pred_r) ** 2)
        ss_tot_r = np.sum((cross_arr_r - np.mean(cross_arr_r)) ** 2)
        r2_r = 1.0 - ss_res_r / ss_tot_r if ss_tot_r != 0 else np.nan

        xs_all_min = min(mem_arr.min() if mem_arr.size > 0 else mem_arr_r.min(), mem_arr_r.min())
        xs_all_max = max(mem_arr.max() if mem_arr.size > 0 else mem_arr_r.max(), mem_arr_r.max())
        xs_r = np.linspace(xs_all_min - 0.2, xs_all_max + 0.2, 200)
        ys_r = slope_r * xs_r + intercept_r
        l2, = ax_lin.plot(xs_r, ys_r, color='gray', linewidth=1, linestyle=':', zorder=2)
        legends.append((l2, f'Rand fit: $R^2$={r2_r:.3f}, a={slope_r:.2f}'))

    if legends:
        handles, labels = zip(*legends)
        ax_lin.legend(handles, labels, fontsize=9, loc='upper left', frameon=False)

    plt.tight_layout()
    if save:
        plt.savefig(f'figures/{model}_both.pdf', format='pdf', bbox_inches='tight')
    plt.show()

    return crossings

for model in MODELS:
    plot_accessibility_curves(model, fit_sigmoid=True, save=True)
